# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 colorectal cancer survivors dataset using the `mlcroissant` library, fully referencing entities and fields by their `@id` as defined in the Croissant schema.

### Dataset Source
The dataset is described and structured according to the Croissant standard, accessible from this schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and show key information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# The Dataset.metadata property gives access to descriptive dataset information
print(f"Title: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
### 2.1 Record Sets and their `@id`
List available record sets, each uniquely referenced by their `@id`. For each record set, display its name and all its field `@id`s.

In [ ]:
# Get all record sets defined in the Croissant metadata
record_sets = dataset.metadata.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name} - @id: {rs.id}")
    print("    Field @id list:")
    for f in rs.fields:
        print(f"        - {f.name} (@id: {f.id})")

### 2.2 Sampling Records with IDs
Inspect a few sample records from a record set, referencing by record set `@id`.

In [ ]:
# Example: List records for the main tabular data record set.
# Find the RecordSet @id for the main data table:
main_recordset = None
for rs in record_sets:
    if ('Clinicopathological' in rs.name) or ('CRC' in rs.name) or len(rs.fields) >= 5:
        main_recordset = rs
        break

if main_recordset is None:
    # Fallback: select the first record set
    main_recordset = record_sets[0]

main_recordset_id = main_recordset.id
print(f"Main data RecordSet @id: {main_recordset_id}")

print("First sample records:")
for i, record in enumerate(dataset.records(record_set=main_recordset_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction
Load the data table(s) into pandas DataFrames, using the record set `@id` and field `@id`s as references.

In [ ]:
# Prepare a list of all record set @ids for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Retrieve records from each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from RecordSet {record_set_id}")

# Show columns from the main record set
if main_recordset_id in dataframes:
    print(f"Fields (@id as columns) in main record set {main_recordset_id}:")
    print(dataframes[main_recordset_id].columns.tolist())
    dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping to numeric fields by referencing field `@id`s.

In [ ]:
# Select a numeric field to analyze (by @id). We'll automatically pick the first detected numeric field as an example.
main_df = dataframes[main_recordset_id]

# Map display: Show all columns (@id)
print(f"Available fields in main DataFrame (by @id):\n{main_df.columns.tolist()}")

# Try to autodetect an integer/float-like field
numeric_field_id = None
for c in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[c]):
        numeric_field_id = c
        break

if not numeric_field_id:
    # Attempt conversion for columns with numeric-sounding names
    for c in main_df.columns:
        # crude attempt to guess based on id
        if 'age' in c.lower() or 'interval' in c.lower() or 'years' in c.lower():
            try:
                main_df[c] = pd.to_numeric(main_df[c], errors='coerce')
                if pd.api.types.is_numeric_dtype(main_df[c]):
                    numeric_field_id = c
                    break
            except:
                continue

if not numeric_field_id:
    raise ValueError('No numeric field found in main record set.')

print(f"\nUsing numeric field for analysis: {numeric_field_id}")

# Threshold: Mean for demonstration; filter for values greater than mean
threshold = main_df[numeric_field_id].mean()
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (by @id), e.g., sex or cancer type
group_field_id = None
candidate_groups = [c for c in main_df.columns if ('sex' in c.lower()) or ('type' in c.lower()) or ('location' in c.lower())]
if candidate_groups:
    group_field_id = candidate_groups[0]

if group_field_id is not None and group_field_id in filtered_df:
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
    )
    print(f"\nGrouped data by {group_field_id} and mean of {numeric_field_id}:")
    print(grouped_df.head())
else:
    print("\nNo suitable grouping field found by @id.")

## 5. Visualization
Visualize distribution of the selected numeric field, optionally grouped by another field, using matplotlib and pandas. All axis labels use field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of the chosen numeric field
plt.figure(figsize=(7, 4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field_id is available, show boxplot
if group_field_id is not None and group_field_id in main_df:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 colorectal cancer survivors dataset using the `mlcroissant` library, referenced all record sets and fields by their `@id`, and explored a numeric and categorical field. We performed filtering, normalization, and visualized distributions, demonstrating a reproducible processing flow that aligns with the dataset's Croissant schema.

**Key takeaways:**
- All steps referenced entities by `@id` as required by Croissant.
- This workflow can be extended to other record sets, fields, or for downstream clinical modeling and research tasks.